In [ ]:
# imports

# Auto-reload modules before executing code
%load_ext autoreload
%autoreload 2

import echopype as ep
from pathlib import Path
import numpy as np


from aa_si_calibration import calibration
from aa_si_calibration import standardized_file_lib
from aa_si_calibration import manufacturer_file_parsers
from aa_si_calibration import comparison

from aa_si_visualization import assorted
from aa_si_ml import ml
from aa_si_utils import utils
import pyvista as pv
from aa_si_visualization.paraview import sv_dataset_to_vtk

import xarray as xr
xr.__version__

In [ ]:
# User defined variables
# NOTE: must include a .bot file with the same name in the same directory as the raw file!
raw_folder_string = './raw_files/' # NOTE: this is now a folder and not a full path to a single raw file
raw_file_names = ["D20090405-T114914.raw"] # Only use if populated with a list strings of raw file names
netcdf_output_folder_string = './NetCDF-files'
sv_output_folder_string = './Sv-files'
cal_folder_string = '../calibration_files/HB200901_cal'
output_logs_folder_string = './Output-Logs'
clear_previous_json_logs = True
standardized_calibration_from_report = "D20090405_cal_report.yml"
standardized_calibration_from_nc = "D20090405_cal_nc_params.yml"
standardized_cal_files_location = "./standardized_cal_files"

standardized_cal_folder = Path(standardized_cal_files_location)
raw_folder = Path(raw_folder_string)
netcdf_output_folder = Path(netcdf_output_folder_string)
netcdf_output_path = netcdf_output_folder / (raw_folder.stem + '.nc') # NOTE: need to calculate this for every raw file if multiple files
sv_output_folder = Path(sv_output_folder_string)
cal_folder = Path(cal_folder_string)
output_logs_folder = Path(output_logs_folder_string)

sv_flag_thresholds = {
                "critical_median": 2.0,
                "large_median": 1.0,
                "moderate_median": 0.5,
                "critical_max": 4.0,
                "large_max": 2.0,
                "moderate_max": 1.0
            }

# Get list of raw files to process
if raw_file_names is not None and len(raw_file_names) > 0:
    # Use specified file names
    raw_file_paths = [raw_folder / filename for filename in raw_file_names]
else:
    # Get all .raw files in folder
    raw_file_paths = sorted(raw_folder.glob("*.raw"))

if not raw_file_paths:
    raise Exception("No raw files found to process")

print(f"Found {len(raw_file_paths)} raw file(s) to process:")
for path in raw_file_paths:
    print(f"  - {path.name}")

if not cal_folder.exists():
    raise Exception("Calibration folder not found")

if not raw_folder.exists():
    raise Exception("Raw folder not found")

# Create output folders if they don't exist
netcdf_output_folder.mkdir(parents=True, exist_ok=True)
sv_output_folder.mkdir(parents=True, exist_ok=True)
output_logs_folder.mkdir(parents=True, exist_ok=True)
standardized_cal_folder.mkdir(parents=True, exist_ok=True)

if clear_previous_json_logs:
    flags_file = Path(output_logs_folder) / "calibration_flags_NEFSC_use_case_1.json"
    if flags_file.exists():
        flags_file.unlink()

In [ ]:
# Open the raw files and convert to netCDF
# Processing Level 1

# Convert each raw file to intermediate netCDF
echodata_nc_list = []
echodata_raw_list = []
netcdf_paths = []
seafloor_depth_list = []  # Store seafloor data separately

for raw_path in raw_file_paths:
    print(f"\nProcessing: {raw_path.name}")

    # Generate netCDF path for this raw file
    netcdf_path = netcdf_output_folder / f"{raw_path.stem}.nc"
    netcdf_paths.append(netcdf_path)

    # Open raw file and convert to netCDF
    ed_elem = ep.open_raw(raw_path, sonar_model="EK60", include_bot=True)
    echodata_raw_list.append(ed_elem)

    # Save to netCDF
    ed_elem.to_netcdf(save_path=netcdf_path)
    print(f"  Converted and saved to: {netcdf_path}")

    # Open converted file and add to list
    echodata_single = ep.open_converted(netcdf_path)
    echodata_nc_list.append(echodata_single)

# Combine all echodata objects if there are multiple files
if len(echodata_nc_list) > 1:
    print(f"\nCombining {len(echodata_nc_list)} echodata objects...")
    echodata = ep.combine_echodata(echodata_nc_list)

else:
    # Only one file, use it directly
    echodata = echodata_raw_list[0] # NOTE: For some reason, the seafloor data is missing in the nc file, so using raw echodata object instead
    print(f"\nUsing single file")

# Verify combined seafloor data
utils.check_for_seafloor_depth_data(echodata)
print(f"\nFinal echodata ready for processing")

In [ ]:
echodata["Sonar/Beam_group1"].channel.values

In [ ]:
# extract calibration parameters from original netcdf file
params = calibration.extract_netcdf_calibration_parameters(echodata, output_logs_folder)

original_cal_params = params["cal_params"]
original_env_params = params["env_params"]
original_other_params = params["other_params"]

# the calibration parameters come from the first file in the sequential raw files that were combined by echopype
nc_cal_files = [str(raw_file_paths[0].name)]

original_other_params["source_filenames_across_channels"] = nc_cal_files
original_other_params["source_file_type"] = ".raw"

In [ ]:
# print parameters from netcdf file
calibration.print_calibration_values(echodata, original_env_params, original_cal_params, original_other_params, "Calibration Values From .nc File")

In [ ]:
report_params = manufacturer_file_parsers.extract_calibration_params_from_EK60_report(
    cal_folder,
    original_other_params["frequency_nominal"],
    output_logs_folder
    )

report_cal_params, report_env_params, report_other_params = manufacturer_file_parsers.convert_ek60_params_to_pipeline_format(report_params)

In [ ]:
# print file calibration parameters
calibration.print_calibration_values(echodata, report_env_params, report_cal_params, report_other_params, "Calibration Values From .cal File Report in .nc format")

In [ ]:
global_params_report = {
    "cruise_id" : "HB0901"
}

file_path_report = standardized_cal_folder / standardized_calibration_from_report

standardized_file_lib.save_cal_params_to_standardized_file(report_cal_params, report_env_params, report_other_params, global_params_report, file_path_report)

In [ ]:
global_params_nc = {
    "cruise_id" : "HB0901"
}

file_path_nc = standardized_cal_folder / standardized_calibration_from_nc

standardized_file_lib.save_cal_params_to_standardized_file(original_cal_params, original_env_params, original_other_params, global_params_nc, standardized_cal_file_path=file_path_nc)

In [ ]:
# Compare the two sets of calibration parameters
comparison.compare_calibration_parameters(report_cal_params, report_env_params, report_other_params, original_cal_params, original_env_params, original_other_params, echodata)

In [ ]:
# Calculate Baseline Sv and Apply seafloor mask:

ds_Sv = ep.calibrate.compute_Sv(echodata)  # obtain a dataset containing Sv, echo_range, and

# Create mask using the corrected function with both raw and calibrated data
mask_blank = utils.createSvMask(ds_Sv)

# Remove seafloor data (and below) with a buffer of 10 m
mask_no_seafloor = utils.remove_seafloor_from_mask(echodata, ds_Sv, mask_blank, buffer_m=10.0)

# remove surface data down to 10 m
mask = utils.remove_surface_from_mask(ds_Sv, mask_no_seafloor, depth_threshold_m=10.0)

utils.log_mask_stats(mask)

ds_Sv_baseline = utils.apply_mask_to_sv(ds_Sv, mask)

# Save the masked dataset
ds_Sv_baseline.to_netcdf(sv_output_folder / "original_processed_data.nc")
print("Seafloor mask applied and saved")

In [ ]:
# Generate Sv with individual parameters from calibration file
# Processing Level 2

# gain correction
cal_params_gain_only = {'gain_correction': report_cal_params["gain_correction"]}
ds_Sv_gain_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_gain_only)
ds_Sv_gain_test = utils.apply_mask_to_sv(ds_Sv_gain_test_unmasked, mask)

# sa correction
cal_params_sa_only = {'sa_correction': report_cal_params["sa_correction"]}
ds_Sv_sa_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_sa_only)
ds_Sv_sa_test = utils.apply_mask_to_sv(ds_Sv_sa_test_unmasked, mask)

# equivalent beam angle
cal_params_eba_only = {'equivalent_beam_angle': report_cal_params["equivalent_beam_angle"]}
ds_Sv_eba_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_eba_only)
ds_Sv_eba_test = utils.apply_mask_to_sv(ds_Sv_eba_test_unmasked, mask)

# equivalent beam angle
env_params_ss_only = {'sound_speed': report_env_params["sound_speed"]}
ds_Sv_ss_test_unmasked = ep.calibrate.compute_Sv(echodata, env_params=env_params_ss_only)
ds_Sv_sound_speed_test = utils.apply_mask_to_sv(ds_Sv_ss_test_unmasked, mask)

# absorption
env_params_ab_only = {'sound_absorption': report_env_params["sound_absorption"]}
ds_Sv_ab_test_unmasked = ep.calibrate.compute_Sv(echodata, env_params=env_params_ab_only)
ds_Sv_absorption_test = utils.apply_mask_to_sv(ds_Sv_ab_test_unmasked, mask)

# Generate Sv with all parameters from calibration file combined
cal_params_combined = {
    'gain_correction': report_cal_params["gain_correction"],
    'sa_correction': report_cal_params["sa_correction"],
    'equivalent_beam_angle': report_cal_params["equivalent_beam_angle"]
    }

env_params_combined = {
    'sound_speed': report_env_params["sound_speed"],
    'sound_absorption': report_env_params["sound_absorption"]
    }

ds_Sv_combined_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_combined, env_params=env_params_combined)
ds_Sv_combined_test = utils.apply_mask_to_sv(ds_Sv_combined_test_unmasked, mask)

In [ ]:
# calculate and print the effect of applying each calibration parameter as well as the combined effect

gain_results = comparison.calculate_full_dataset_effect(ds_Sv_gain_test, ds_Sv_baseline, "Gain Correction", output_logs_folder, sv_flag_thresholds)

sa_results = comparison.calculate_full_dataset_effect(ds_Sv_sa_test, ds_Sv_baseline, "Sa Correction", output_logs_folder, sv_flag_thresholds)

eba_results = comparison.calculate_full_dataset_effect(ds_Sv_eba_test, ds_Sv_baseline, "Equivalent Beam Angle", output_logs_folder, sv_flag_thresholds)

sound_speed_results = comparison.calculate_full_dataset_effect(ds_Sv_sound_speed_test, ds_Sv_baseline, "Sound Speed", output_logs_folder, sv_flag_thresholds)

absorption_results = comparison.calculate_full_dataset_effect(ds_Sv_absorption_test, ds_Sv_baseline, "Absorption", output_logs_folder, sv_flag_thresholds)

combined_results = comparison.calculate_full_dataset_effect(ds_Sv_combined_test, ds_Sv_baseline, "Combined Results", output_logs_folder, sv_flag_thresholds)

In [ ]:
# plot effects of each calibration parameter

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_absorption_test,
    "Absorption Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_gain_test,
    "Gain Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_sa_test,
    "Sa Correction Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_sound_speed_test,
    "Sound Speed Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_combined_test,
    "Combined Differences"
)

In [ ]:
# Perform range-dependent analysis with Absorption results
comparison.perform_range_analysis(ds_Sv_baseline, ds_Sv_absorption_test, echodata, "Absorption Effect Range Analysis")

In [ ]:
# VISUALIZATION OF CALIBRATION DIFFERENCES

min_depth = 20
max_depth = 220
ping_min = 100
ping_max = 300

assorted.sv_differences_echograms(ds_Sv_baseline, ds_Sv_combined_test, original_other_params["frequency_nominal"], max_depth, min_depth, ping_min, ping_max, x_axis_units="pings", y_axis_units="meters", meters_per_second=1)

In [ ]:
# Verify that individual parameter effects add up to the combined effect
verification_results = comparison.verify_additive_effects(
    gain_results,
    sa_results,
    eba_results,
    sound_speed_results,
    absorption_results,
    combined_results
)

In [ ]:
ds_Sv_clean, ds_MVBS = ml.data_preprocessing_pipeline(
    ds_Sv_combined_test,
    echodata,
    noise_range_sample_num=10,
    noise_ping_num=5,
    mvbs_range_bin="2m",
    mvbs_ping_time_bin="2s",
    mvbs_nan_threshold=0.6,
    plot_window=[0, 200, 0, 1000],
    y_to_x_aspect_ratio_override=3
    )

In [ ]:
cluster_colors = [
                "#FF00EA", "#11FF00", "#005EFF", "#2F00FF", "#FF2DF4",
                "#FFFB1C", "#4E9200", "#970021", "#8E00E0", "#017685FF", "#4100B9FF"
            ]

In [ ]:
ml.full_dbscan_iteration(
    ds_MVBS,
    custom_dataset_name="ml_dataset_5",
    feature_strategy="mean_centered",
    # baseline_channel=3,
    data_var="Sv",
    custom_normalization_name="normalized_data_quantile_mc_2",
    ml_result_name="dbscan_results",
    normalization_strategy="flatten",
    feature_weights=[0, 1, 1, 1, 1],
    plot_window=[0, 200, 0, 1000],
    min_samples=8,
    sample_size=200000,
    min_cluster_size=400,
    cluster_selection_method="leaf",
    use_hdbscan=True,
    y_to_x_aspect_ratio_override=3,
    # soft_membership_threshold=0.7,
    ds_Sv_original=ds_Sv_combined_test,
    cluster_colors=cluster_colors,
    cluster_stats_sv_data_var="Sv",
    cluster_stats_compute_pairwise_differences=False
    )

In [ ]:
print(ds_MVBS)

In [ ]:
# Ensure pvxarray accessor is registered
try:
    import pvxarray.accessor
    from pvxarray.accessor import PyVistaAccessor
    xr.register_dataset_accessor("pyvista")(PyVistaAccessor)
    print("✅ PyVista accessor registered")
except Exception as e:
    print(f"⚠️ PyVista accessor registration failed: {e}")

In [ ]:
# For Paraview: Create rectilinear grid with ALL channels as separate cell data arrays

# Get data for ALL channels
sv_data_all_channels = ds_Sv_combined_test["Sv"].values  # Shape: (channel, ping_time, range_sample)
ping_times = ds_Sv_combined_test["ping_time"].values
range_samples = ds_Sv_combined_test["range_sample"].values
echo_range = ds_Sv_combined_test["echo_range"].isel(channel=0).values  # Assuming same for all channels
frequencies = ds_Sv_combined_test["frequency_nominal"].values

print(f"All channels data shape: {sv_data_all_channels.shape}")
print(f"Number of channels: {sv_data_all_channels.shape[0]}")

# Convert ping_time to numeric (seconds from start)
time_numeric = (ping_times - ping_times[0]).astype('timedelta64[s]').astype(float)

# Find valid data bounds using first channel (assuming similar coverage)
sv_c1 = sv_data_all_channels[0]  # First channel for bounds calculation
valid_mask = ~np.isnan(sv_c1)
valid_ping_indices, valid_range_indices = np.where(valid_mask)

# Find bounding box of valid data
ping_min_idx = valid_ping_indices.min()
ping_max_idx = valid_ping_indices.max()
range_min_idx = valid_range_indices.min()
range_max_idx = valid_range_indices.max()

# Crop all channels to the same bounding box
sv_data_all_cropped = sv_data_all_channels[:, ping_min_idx:ping_max_idx+1, range_min_idx:range_max_idx+1]
time_cropped = time_numeric[ping_min_idx:ping_max_idx+1]
range_indices_cropped = range_samples[range_min_idx:range_max_idx+1]

# Use actual depths from first channel (assuming consistent across channels)
echo_range_cropped = echo_range[ping_min_idx:ping_max_idx+1, range_min_idx:range_max_idx+1]
depth_coords = echo_range_cropped[0, :]

print(f"Coordinate ranges:")
print(f"  Time: {time_cropped.min():.2f} to {time_cropped.max():.2f} seconds")
print(f"  Depth: {depth_coords.min():.1f} to {depth_coords.max():.1f} meters")

# Create coordinate arrays
x_coords = np.concatenate([time_cropped, [time_cropped[-1] + (time_cropped[-1] - time_cropped[-2])]])
y_coords = np.concatenate([-depth_coords, [-depth_coords[-1] - (depth_coords[-1] - depth_coords[-2])]])
z_coords = np.array([0])

# Create the rectilinear grid (same structure for all channels)
grid_all_channels = pv.RectilinearGrid(x_coords, y_coords, z_coords)

# Add each channel as a separate cell data array
for ch_idx in range(sv_data_all_channels.shape[0]):
    sv_channel_data = sv_data_all_cropped[ch_idx]
    sv_flat_channel = sv_channel_data.T.flatten()

    # Handle size mismatch if needed
    if sv_flat_channel.size != grid_all_channels.n_cells:
        if grid_all_channels.n_cells > sv_flat_channel.size:
            padding_needed = grid_all_channels.n_cells - sv_flat_channel.size
            sv_flat_channel = np.concatenate([sv_flat_channel, np.full(padding_needed, np.nan)])

    # Add channel data with descriptive name
    freq_khz = int(frequencies[ch_idx] / 1000)
    channel_name = f"Sv_dB_Ch{ch_idx}_{freq_khz}kHz"
    grid_all_channels.cell_data[channel_name] = sv_flat_channel

    print(f"Added channel {ch_idx}: {channel_name}")

# Add comprehensive metadata
grid_all_channels.field_data["n_channels"] = np.array([sv_data_all_channels.shape[0]])
grid_all_channels.field_data["frequencies_hz"] = frequencies
grid_all_channels.field_data["time_range_seconds"] = np.array([x_coords[0], x_coords[-1]])
grid_all_channels.field_data["depth_range_meters"] = np.array([y_coords[0], y_coords[-1]])
grid_all_channels.field_data["aspect_ratio_m_per_s"] = np.array([(y_coords[-1]-y_coords[0])/(x_coords[-1]-x_coords[0])])

# Save single file with all channels
output_file_all_channels = sv_output_folder / "sv_all_channels_grid_2.vtk"
grid_all_channels.save(output_file_all_channels, binary=True)

print(f"\n✅ Saved all {sv_data_all_channels.shape[0]} channels to: {output_file_all_channels}")
print(f"Each channel is available as a separate cell data array")
print(f"Channel names: {list(grid_all_channels.cell_data.keys())}")

In [ ]:
# Analyze data distribution to find optimal DBSCAN parameters
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

def analyze_data_for_dbscan(X_normalized, sample_size=10000):
    """
    Analyze data to suggest optimal DBSCAN parameters
    """
    print("=== ANALYZING DATA FOR OPTIMAL DBSCAN PARAMETERS ===")

    # Use a sample for computational efficiency
    if len(X_normalized) > sample_size:
        np.random.seed(42)
        sample_idx = np.random.choice(len(X_normalized), size=sample_size, replace=False)
        X_sample = X_normalized[sample_idx]
        print(f"Using sample of {sample_size:,} points for analysis")
    else:
        X_sample = X_normalized
        print(f"Using all {len(X_normalized):,} points for analysis")

    print(f"\nData distribution analysis:")
    print(f"  Shape: {X_sample.shape}")
    print(f"  Value range: {X_sample.min():.3f} to {X_sample.max():.3f}")
    print(f"  Mean: {X_sample.mean():.3f}")
    print(f"  Std: {X_sample.std():.3f}")

    # Calculate pairwise distances to understand data density
    print(f"\nCalculating nearest neighbor distances...")

    # Use k=4 (a common choice for min_samples in DBSCAN)
    k = 4
    nbrs = NearestNeighbors(n_neighbors=k+1)  # +1 because first neighbor is the point itself
    nbrs.fit(X_sample)
    distances, indices = nbrs.kneighbors(X_sample)

    # Get k-th nearest neighbor distances (excluding self)
    k_distances = distances[:, k]
    k_distances = np.sort(k_distances)

    print(f"K-distance statistics (k={k}):")
    print(f"  Min k-distance: {k_distances.min():.4f}")
    print(f"  Max k-distance: {k_distances.max():.4f}")
    print(f"  Mean k-distance: {k_distances.mean():.4f}")
    print(f"  Median k-distance: {np.median(k_distances):.4f}")
    print(f"  90th percentile: {np.percentile(k_distances, 90):.4f}")
    print(f"  95th percentile: {np.percentile(k_distances, 95):.4f}")
    print(f"  99th percentile: {np.percentile(k_distances, 99):.4f}")

    # Create k-distance plot to find the "elbow"
    plt.figure(figsize=(12, 8))

    # Plot 1: K-distance plot (sorted)
    plt.subplot(2, 2, 1)
    plt.plot(k_distances, 'b-', linewidth=1)
    plt.xlabel('Points sorted by distance')
    plt.ylabel(f'{k}-distance')
    plt.title(f'{k}-Distance Plot (Elbow Method)')
    plt.grid(True, alpha=0.3)

    # Add horizontal lines for percentiles
    for percentile, color, alpha in [(90, 'orange', 0.7), (95, 'red', 0.7), (99, 'darkred', 0.7)]:
        value = np.percentile(k_distances, percentile)
        plt.axhline(y=value, color=color, linestyle='--', alpha=alpha,
                   label=f'{percentile}th percentile: {value:.4f}')
    plt.legend()

    # Plot 2: Histogram of k-distances
    plt.subplot(2, 2, 2)
    plt.hist(k_distances, bins=50, alpha=0.7, edgecolor='black')
    plt.xlabel(f'{k}-distance')
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {k}-distances')
    plt.grid(True, alpha=0.3)

    # Plot 3: Zoomed k-distance plot (focus on elbow region)
    plt.subplot(2, 2, 3)
    # Focus on the region where we expect to find the elbow
    focus_end = int(len(k_distances) * 0.95)  # Look at first 95% of points
    plt.plot(k_distances[:focus_end], 'b-', linewidth=1)
    plt.xlabel('Points sorted by distance (first 95%)')
    plt.ylabel(f'{k}-distance')
    plt.title(f'{k}-Distance Plot (Zoomed)')
    plt.grid(True, alpha=0.3)

    # Plot 4: Feature correlation matrix
    plt.subplot(2, 2, 4)
    corr_matrix = np.corrcoef(X_sample.T)
    im = plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im)
    plt.title('Feature Correlation Matrix')

    # Add frequency labels if available
    if 'frequency_info' in globals():
        freq_labels = [f"{label}" for label in frequency_info['frequency_labels']]
        plt.xticks(range(len(freq_labels)), freq_labels, rotation=45)
        plt.yticks(range(len(freq_labels)), freq_labels)

    plt.tight_layout()
    plt.show()

    # Suggest eps values based on k-distance analysis
    print(f"\n=== SUGGESTED DBSCAN PARAMETERS ===")

    # Common approaches for choosing eps:
    # 1. Around the "elbow" in k-distance plot
    # 2. 90th-95th percentile of k-distances
    # 3. Mean + 1-2 standard deviations

    suggested_eps = []

    # Method 1: Percentile-based
    eps_90 = np.percentile(k_distances, 90)
    eps_95 = np.percentile(k_distances, 95)
    suggested_eps.extend([eps_90, eps_95])

    # Method 2: Statistical
    mean_dist = k_distances.mean()
    std_dist = k_distances.std()
    eps_mean_plus_std = mean_dist + std_dist
    eps_mean_plus_2std = mean_dist + 2*std_dist
    suggested_eps.extend([eps_mean_plus_std, eps_mean_plus_2std])

    # Method 3: Elbow detection (simplified)
    # Look for the point where the slope changes dramatically
    diff1 = np.diff(k_distances)
    diff2 = np.diff(diff1)

    # Find points with high second derivative (sharp turns)
    elbow_candidates = np.where(diff2 > np.percentile(diff2, 95))[0]
    if len(elbow_candidates) > 0:
        # Take the first significant elbow
        elbow_idx = elbow_candidates[0]
        eps_elbow = k_distances[elbow_idx]
        suggested_eps.append(eps_elbow)

    # Remove duplicates and sort
    suggested_eps = sorted(list(set([round(eps, 4) for eps in suggested_eps])))

    print("Suggested eps values to try:")
    for i, eps in enumerate(suggested_eps):
        method = []
        if abs(eps - eps_90) < 0.0001:
            method.append("90th percentile")
        if abs(eps - eps_95) < 0.0001:
            method.append("95th percentile")
        if abs(eps - eps_mean_plus_std) < 0.0001:
            method.append("mean + 1σ")
        if abs(eps - eps_mean_plus_2std) < 0.0001:
            method.append("mean + 2σ")
        if 'eps_elbow' in locals() and abs(eps - eps_elbow) < 0.0001:
            method.append("elbow detection")

        method_str = ", ".join(method) if method else "derived"
        print(f"  {i+1}. eps = {eps:.4f} ({method_str})")

    print(f"\nSuggested min_samples values:")
    print(f"  - For dense regions: 5-10")
    print(f"  - For moderate density: 10-20")
    print(f"  - For sparse regions: 3-5")
    print(f"  - Rule of thumb: 2 × dimensionality = 2 × {X_sample.shape[1]} = {2 * X_sample.shape[1]}")

    # Check if current parameters make sense
    current_eps = 0.5
    print(f"\n=== ANALYSIS OF CURRENT PARAMETERS ===")
    print(f"Current eps = {current_eps}")
    print(f"This is {'MUCH LARGER' if current_eps > eps_95 else 'larger' if current_eps > eps_90 else 'reasonable compared'} than suggested values")
    print(f"95th percentile of distances: {eps_95:.4f}")
    print(f"Your eps ({current_eps}) is {current_eps/eps_95:.1f}x the 95th percentile")

    if current_eps > eps_95:
        print(f"❌ Problem: eps is too large - this explains why you get only 1 cluster")
        print(f"✅ Solution: Try much smaller eps values: {suggested_eps[:3]}")

    return suggested_eps, k_distances

# Run the analysis
suggested_eps_values, k_distances = analyze_data_for_dbscan(X_standard, sample_size=10000)

In [ ]:
# Test DBSCAN with the suggested optimal parameters
print("=== TESTING DBSCAN WITH OPTIMAL PARAMETERS ===")

# Use the suggested eps values from the analysis
optimal_eps_values = [0.3, 0.4, 0.45]    # Start from very small and work up
optimal_min_samples = [100] # Based on dimensionality and density

# Run optimized DBSCAN with the suggested parameters
print("🚀 Testing DBSCAN with optimal parameters...")
optimal_dbscan_results = apply_dbscan_clustering(
    X_standard,
    eps_values=optimal_eps_values,
    min_samples_values=optimal_min_samples,
    use_all_data=True,
    calculate_silhouette=True,
    silhouette_sample_size=10000
)

In [ ]:
# Analysis of current DBSCAN results and refined parameter testing

print("=== REFINED PARAMETER TESTING ===")
print("Let's test around the sweet spot we found...")

# Test refined parameters around the promising eps=0.4 range
refined_eps_values = [0.4]  # Around the sweet spot
refined_min_samples = [100]  # Test different densities around optimal

print("🔬 Testing refined parameters around the optimal range...")
refined_dbscan_results = apply_dbscan_clustering(
    X_standard,
    eps_values=refined_eps_values,
    min_samples_values=refined_min_samples,
    use_all_data=True,
    calculate_silhouette=True,
    silhouette_sample_size=10000
)

In [ ]:
# Test the promising middle range around eps=0.25
print("=== CORRECTED ANALYSIS ===")
print("I made an error - eps=0.4 gives 1 cluster, not 3!")
print("From your earlier results, eps=0.25 gave 6 clusters with 1.9% noise and positive silhouette.")
print("Let's test around that promising range...")
print()

# Test parameters around the actually promising eps=0.25 range
promising_eps_values = [0.2, 0.25, 0.3, 0.35]  # Around eps=0.25 which worked
promising_min_samples = [5, 8, 10, 15]  # Different density thresholds

print("🎯 Testing around eps=0.25 which actually showed promise...")
promising_dbscan_results = apply_dbscan_clustering(
    X_standard,
    eps_values=promising_eps_values,
    min_samples_values=promising_min_samples,
    use_all_data=True,
    calculate_silhouette=True,
    silhouette_sample_size=10000
)